# Практикум 16. Большие языковые модели: генерация текста и проектирование подсказок

В лаборатории 16 вы двигали ползунки температуры, top-k и top-p, сравнивали подсказки zero-shot, few-shot и chain-of-thought и смотрели, как голосование нескольких рассуждений (self-consistency) повышает точность. Здесь сделаем то же самое в коде:

1. реализуем с нуля greedy, температуру, top-k и top-p и проверим их свойствами;
2. обучим биграммную языковую модель и сравним стратегии генерации: разнообразие против качества;
3. соберём шаблоны подсказок zero-shot / few-shot / CoT и научимся извлекать и проверять ответ;
4. посчитаем точность голосования self-consistency по биномиальной формуле и симуляцией;
5. в Colab попробуем всё это на небольшой настоящей модели.

*Практикум к лаборатории 16 курса «Алгоритмы NLP». Запускайте ячейки сверху вниз: Shift+Enter или «Среда выполнения → Выполнить всё». Чтобы сохранить свои ответы, сделайте «Файл → Сохранить копию на Диске».*

## 1. Декодирование: от логитов к следующему токену

На каждом шаге языковая модель выдаёт вектор **логитов** $z$ — по одному числу на каждый токен словаря. Как выбрать следующий токен?

- **greedy** (жадно): $\arg\max_i z_i$ — детерминированно, но однообразно;
- **температура**: $p_i = \dfrac{e^{z_i/T}}{\sum_j e^{z_j/T}}$. При $T<1$ распределение заостряется, при $T>1$ сглаживается, при $T\to 0$ получается greedy;
- **top-k**: оставить $k$ самых вероятных токенов и перенормировать;
- **top-p** (nucleus): оставить наименьшее множество самых вероятных токенов с суммарной вероятностью $\ge p$ и перенормировать. Размер «ядра» меняется в зависимости от уверенности модели.

Возьмём игрушечный вектор логитов для продолжения фразы «На завтрак я обычно пью …».

In [ ]:
import numpy as np

context = "На завтрак я обычно пью"
vocab = ["чай", "кофе", "воду", "молоко", "сок", "какао", "кефир", "компот", "айран", "лимонад"]
logits = np.array([3.2, 2.9, 1.8, 1.5, 1.2, 0.6, 0.3, -0.2, -0.5, -1.0])   # выход модели для следующего слова


def softmax_t(logits, T=1.0):
    # softmax с температурой T
    z = np.asarray(logits, dtype=float) / T
    e = np.exp(z - z.max())
    return e / e.sum()

def entropy(p):
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum()) + 0.0       # + 0.0 убирает «-0.00» у greedy

def greedy(p):
    q = np.zeros_like(p, dtype=float)
    q[np.argmax(p)] = 1.0
    return q

def top_k_filter(p, k):
    # оставить k самых вероятных токенов
    keep = np.argsort(-p, kind="stable")[:k]
    q = np.zeros_like(p)
    q[keep] = p[keep]
    return q / q.sum()

def top_p_filter(p, top_p):
    # наименьшее множество токенов с суммой вероятностей >= top_p
    order = np.argsort(-p, kind="stable")
    cum = np.cumsum(p[order])
    n = min(int(np.searchsorted(cum, top_p)) + 1, len(p))
    q = np.zeros_like(p)
    q[order[:n]] = p[order[:n]]
    return q / q.sum()

def decode_dist(logits, T=1.0, k=None, top_p=None):
    # порядок как в Hugging Face: температура -> top-k -> top-p; T=0 означает greedy
    if T == 0:
        return greedy(np.asarray(logits, dtype=float))
    q = softmax_t(logits, T)
    if k is not None:
        q = top_k_filter(q, k)
    if top_p is not None:
        q = top_p_filter(q, top_p)
    return q

settings = {"T=1": dict(T=1.0), "T=0.5": dict(T=0.5), "T=2": dict(T=2.0),
            "top-k=3": dict(k=3), "top-p=0.9": dict(top_p=0.9), "greedy": dict(T=0)}
dists = {name: decode_dist(logits, **kw) for name, kw in settings.items()}

print(context, "...\n")
print(f"{'':<10}{'logit':>6}" + "".join(f"{n:>11}" for n in dists))
for i, w in enumerate(vocab):
    print(f"{w:<10}{logits[i]:6.1f}" + "".join(f"{d[i]:11.3f}" for d in dists.values()))
print(f"{'энтропия, бит':<16}" + "".join(f"{entropy(d):11.2f}" for d in dists.values()))

Проверим реализацию свойствами, которые должны выполняться всегда: top-k с $k=1$ и top-p с очень маленьким $p$ совпадают с greedy; при $T\to 0$ получается greedy, при $T\to\infty$ — равномерное распределение; температура не меняет порядок токенов. Затем убедимся, что сэмплирование действительно воспроизводит распределение.


На графике серые столбики — исходное распределение при $T=1$, синие — после преобразования. Обрезанные top-k и top-p токены остаются серыми: их вероятность стала нулевой, а вероятность оставшихся выросла.

In [ ]:
p1 = softmax_t(logits)
assert all(np.isclose(d.sum(), 1.0) for d in dists.values())        # все распределения суммируются в 1
assert np.allclose(top_k_filter(p1, 1), greedy(p1))                 # top-k с k=1 — это greedy
assert np.allclose(top_p_filter(p1, 1e-9), greedy(p1))              # top-p с крошечным p — тоже greedy
assert np.allclose(top_k_filter(p1, len(p1)), p1)                   # k = |V| и p = 1 ничего не меняют
assert np.allclose(top_p_filter(p1, 1.0), p1)
assert np.allclose(softmax_t(logits, 0.01), greedy(p1), atol=1e-6)  # T -> 0: greedy
assert np.allclose(softmax_t(logits, 1000), 1 / len(p1), atol=1e-2) # T -> бесконечность: равномерное распределение
assert all(np.argmax(softmax_t(logits, T)) == np.argmax(logits) for T in [0.1, 1, 10])  # температура не меняет самый вероятный токен
print("Все проверки пройдены.\n")

for T in [0.25, 0.5, 1, 2, 4]:
    q = softmax_t(logits, T)
    n_core = int((top_p_filter(q, 0.9) > 0).sum())
    print(f"T={T:<5} H={entropy(q):.2f} бит   p({vocab[0]})={q[0]:.2f}   ядро top-p 0.9, токенов = {n_core}")

rng = np.random.default_rng(0)
q = dists["top-p=0.9"]
draws = rng.choice(len(vocab), size=20000, p=q)
freq = np.bincount(draws, minlength=len(vocab)) / len(draws)
print("\nСэмплирование top-p=0.9, 20000 раз: вероятность и частота")
for w, a, b in zip(vocab, q, freq):
    print(f"   {w:<10} p = {a:.3f}   частота = {b:.3f}")


import matplotlib.pyplot as plt

show = ["T=1", "T=0.5", "T=2", "top-k=3", "top-p=0.9"]
x = np.arange(len(vocab))
fig, axes = plt.subplots(1, len(show), figsize=(15, 3.2), sharey=True)
for ax, name in zip(axes, show):
    d = dists[name]
    ax.bar(x, p1, color="#DDDDDD", label="T=1")                       # серым — исходное распределение при T=1
    ax.bar(x, d, width=0.55, color=["#2B55C9" if v > 0 else "#DDDDDD" for v in d], label=name)
    ax.set_title(name)
    ax.set_xticks(x)
    ax.set_xticklabels(vocab, rotation=70, fontsize=8)
axes[0].set_ylabel("вероятность")
fig.suptitle("Стратегии декодирования: «" + context + " …»")
plt.tight_layout()
plt.show()

### Как это выглядит на настоящей модели

Возьмём настоящие логиты модели `Qwen/Qwen2.5-0.5B-Instruct` (0,5 млрд параметров, файл ≈1 ГБ) для той же фразы и применим к ним наши функции. Словарь модели — около 150 тысяч токенов. Посмотрите, сколько из них попадает в «ядро» top-p при разных температурах.

В конце сравним, на сколько токенов делится одно и то же предложение по-русски, по-узбекски и по-английски. Языки, которых в обучающих данных токенизатора было мало, делятся на более мелкие кусочки (см. темы 2 и 17): запрос на таком языке длиннее и дороже. Ячейка работает только в Colab с интернетом.

In [ ]:
# в Colab transformers и torch уже установлены; модель ≈1 ГБ скачивается один раз
try:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    name = "Qwen/Qwen2.5-0.5B-Instruct"                     # 0,5 млрд параметров
    tok = AutoTokenizer.from_pretrained(name)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = "auto" if device == "cuda" else torch.float32     # на CPU float32 быстрее, чем bfloat16
    llm = AutoModelForCausalLM.from_pretrained(name, torch_dtype=dtype).to(device)

    ids = tok(context, return_tensors="pt").to(device)
    with torch.no_grad():
        real_logits = llm(**ids).logits[0, -1].float().cpu().numpy()   # логиты для следующего токена
    print("размер словаря модели:", real_logits.shape[0])
    p_real = softmax_t(real_logits)
    for i in np.argsort(-p_real)[:10]:
        print(f"   {tok.decode([int(i)])!r:<14} {p_real[i]:.3f}")
    for T in [0.5, 1.0, 1.5]:
        q = softmax_t(real_logits, T)
        n_core = int((top_p_filter(q, 0.9) > 0).sum())
        print(f"T={T}: H = {entropy(q):.2f} бит, ядро top-p 0.9, токенов = {n_core}")

    # одно и то же предложение на трёх языках
    same = {"ru": "Завтра мы поедем на поезде из Ташкента в Самарканд.",
            "uz": "Ertaga biz poyezdda Toshkentdan Samarqandga boramiz.",
            "en": "Tomorrow we will take the train from Tashkent to Samarkand."}
    print()
    for lang, s in same.items():
        pieces = [tok.decode([t]) for t in tok(s)["input_ids"]]     # кусок может оказаться половиной буквы (�): токены — это байты UTF-8
        print(f"{lang}: {len(s.split())} слов -> {len(pieces)} токенов: {pieces}")
except Exception as e:
    print("Не удалось запустить модель (нужен интернет в Colab):", e)

## 2. Генерация текста биграммной моделью

Настоящая LLM — это та же схема «логиты → декодирование → токен», повторённая много раз. Возьмём самую простую языковую модель — биграммную (тема 4): $P(w_i\mid w_{i-1})=\dfrac{c(w_{i-1},w_i)}{c(w_{i-1})}$. Логиты — это $\log P(w\mid w_{i-1})$ для слов, которые встречались после $w_{i-1}$ в корпусе из 30 предложений. Сверим наши вероятности с `nltk.lm.MLE`.

In [ ]:
corpus = [
    "мы завтра поедем в самарканд на поезде", "мы завтра поедем в бухару на машине",
    "я завтра поеду в ташкент на поезде", "поезд в самарканд уходит утром",
    "поезд в бухару уходит вечером", "утром мы пьём чай", "вечером мы пьём чай с молоком",
    "в самарканде мы увидим регистан", "в бухаре мы увидим старый город", "старый город очень красивый",
    "регистан очень красивый", "я люблю зелёный чай", "я люблю старый город", "мы купили билеты на поезд",
    "билеты на поезд стоят недорого", "в самарканде очень жарко", "в бухаре летом очень жарко",
    "вечером в городе прохладно", "утром в городе прохладно", "я привёз из самарканда лепёшки",
    "мы привезли из бухары подарки", "самаркандские лепёшки очень вкусные", "этот чай очень вкусный",
    "мама утром печёт лепёшки", "мама вечером готовит плов", "мы вечером едим плов", "я люблю плов",
    "в ташкенте есть метро", "в ташкенте мы ездим на метро", "поезд приходит в ташкент вечером",
]
PREV_DEMO = "я"


import re
from collections import Counter, defaultdict

def tokenize(s):
    return re.findall(r"[^\W\d_]+", s.lower())

bigrams = defaultdict(Counter)
for s in corpus:
    toks = ["<s>"] + tokenize(s) + ["</s>"]
    for a, b in zip(toks, toks[1:]):
        bigrams[a][b] += 1

def next_logits(prev):
    # логиты = log P(w | prev) для слов, встречавшихся после prev
    words = sorted(bigrams[prev])
    counts = np.array([bigrams[prev][w] for w in words], dtype=float)
    return words, np.log(counts / counts.sum())

words, lp = next_logits(PREV_DEMO)
print(f"P(w | {PREV_DEMO}):")
for T in [1.0, 0.5, 2.0]:
    q = decode_dist(lp, T)
    print(f"  T={T}: " + ", ".join(f"{w} {x:.2f}" for w, x in sorted(zip(words, q), key=lambda t: -t[1])))

# сверка с nltk.lm (без сглаживания, MLE)
from nltk.lm import MLE
from nltk.lm.preprocessing import padded_everygram_pipeline
train_ng, lm_vocab = padded_everygram_pipeline(2, [tokenize(s) for s in corpus])
lm = MLE(2)
lm.fit(train_ng, lm_vocab)
diff = max(abs(np.exp(l) - lm.score(w, [prev]))
           for prev in list(bigrams) for w, l in zip(*next_logits(prev)))
print(f"\nсловарь: {len(lm.vocab) - 1}, разных биграмм: {sum(len(c) for c in bigrams.values())}")
print(f"max |наши - nltk| = {diff:.2e}")

Сгенерируем предложения разными стратегиями. Greedy всегда выдаёт одно и то же; маленькая температура держится за частые пути; большая собирает необычные сочетания.

In [ ]:
def generate(T=1.0, k=None, top_p=None, rng=None, max_len=12):
    # генерация слово за словом; возвращает текст и средний log P по модели с T=1
    prev, out, logps = "<s>", [], []
    while len(out) < max_len:
        words, lp = next_logits(prev)
        q = decode_dist(lp, T, k, top_p)
        i = int(np.argmax(q)) if T == 0 else int(rng.choice(len(words), p=q))
        logps.append(lp[i])
        if words[i] == "</s>":
            break
        out.append(words[i])
        prev = words[i]
    return " ".join(out), float(np.mean(logps))

rng = np.random.default_rng(1)
print("greedy:      ", generate(T=0)[0])
for name, kw in [("T=0.5", dict(T=0.5)), ("T=1", dict(T=1.0)), ("T=3", dict(T=3.0)),
                 ("top-k=2", dict(k=2)), ("top-p=0.6", dict(top_p=0.6))]:
    for _ in range(3):
        print(f"{name:<12}", generate(rng=rng, **kw)[0])
    print()

### Разнообразие против качества

Сгенерируем по 300 предложений для каждой стратегии и посчитаем:

- **разные** — доля различных предложений среди 300 (разнообразие);
- **новые** — доля предложений, которых нет в корпусе (модель «сочиняет», а не повторяет);
- **средний log P** на слово по исходной модели ($T=1$): чем ближе к 0, тем типичнее текст для модели.

Разнообразие быстро растёт от greedy до $Tpprox 1$, дальше почти не меняется: у маленькой биграммной модели просто мало возможных предложений. «Типичность» (средний log P) с ростом $T$ падает. top-k и top-p отрезают редкие продолжения и потому дают более типичный текст, чем чистое сэмплирование при $T=1$. У биграммной модели каждая соседняя пара слов была в корпусе, поэтому локально всё гладко, но модель видит только одно предыдущее слово: согласование и смысл целого предложения теряются («лепёшки очень вкусный», «увидим регистан очень жарко»).

In [ ]:
train_set = {" ".join(tokenize(s)) for s in corpus}

def evaluate(n=300, **kw):
    sents = [generate(rng=rng, **kw) for _ in range(n)]
    distinct = len({s for s, _ in sents}) / n                       # доля различных предложений среди n
    novel = np.mean([s not in train_set for s, _ in sents])
    return distinct, novel, np.mean([l for _, l in sents])

rng = np.random.default_rng(2)
rows = []
print(f"{'':<12}{'разные':>12}{'новые':>10}{'средний log P':>17}")
for name, kw in [("greedy", dict(T=0)), ("T=0.3", dict(T=0.3)), ("T=0.5", dict(T=0.5)),
                 ("T=1", dict(T=1.0)), ("T=1.5", dict(T=1.5)), ("T=2", dict(T=2.0)), ("T=3", dict(T=3.0)),
                 ("top-k=2", dict(k=2)), ("top-p=0.6", dict(top_p=0.6))]:
    dist, nov, alp = evaluate(**kw)
    rows.append((name, dist, nov, alp))
    print(f"{name:<12}{dist:12.2f}{nov:10.2f}{alp:17.2f}")

temp_rows = [r for r in rows if r[0].startswith("T=")]
Ts = [float(r[0][2:]) for r in temp_rows]
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].plot(Ts, [r[1] for r in temp_rows], "o-", label="разные")
ax[0].plot(Ts, [r[2] for r in temp_rows], "s-", label="новые")
ax[0].set_xlabel("температура T"); ax[0].set_title("Разнообразие"); ax[0].legend()
ax[1].plot(Ts, [r[3] for r in temp_rows], "o-", color="#D9822B")
ax[1].set_xlabel("температура T"); ax[1].set_title("Типичность: средний log P на слово")
plt.tight_layout()
plt.show()

## 3. Проектирование подсказок

Подсказка (prompt) — просто строка, с которой начинается генерация. От её вида сильно зависит ответ:

- **zero-shot** — только инструкция и вопрос;
- **few-shot** — перед вопросом несколько решённых примеров (in-context learning: веса модели не меняются, она «учится» по контексту);
- **chain-of-thought (CoT)** — в примерах показаны промежуточные шаги, или в подсказку добавлена фраза «Давай рассуждать шаг за шагом» (zero-shot CoT).

Почему CoT помогает в многошаговой арифметике? Модель тратит на каждый токен одинаковый объём вычислений. Если ответ надо выдать сразу, все шаги приходится «сделать в уме» за один токен. Если записывать шаги, каждый шаг — простая операция, а её результат уже лежит в контексте и используется дальше.

Соберём шаблоны функцией Python. Правильные ответы задач считаем кодом. Обратите внимание на цену: few-shot и CoT заметно длиннее, а платят за токены.

In [ ]:
problems = [
    "В поезде 8 вагонов, в каждом 50 мест. Продано 270 билетов. Сколько мест осталось свободными?",
    "Взрослый билет стоит 120 тысяч сумов, детский — в два раза дешевле. Семья купила 3 взрослых и 2 детских билета. Сколько тысяч сумов заплатила семья?",
    "У Алишера было 5 тетрадей. Он купил ещё 3 упаковки по 4 тетради и подарил брату 6 тетрадей. Сколько тетрадей у него осталось?",
    "В классе 36 учеников. Треть из них поехала на экскурсию в Бухару, остальные разделились поровну на 4 группы. Сколько учеников в каждой группе?",
    "На рынке 1 кг яблок стоит 15 тысяч сумов, а 1 кг винограда на 7 тысяч дороже. Сколько тысяч сумов стоят 2 кг яблок и 3 кг винограда?",
]
# правильные ответы считаем кодом, а не вручную
gold = [8 * 50 - 270, 3 * 120 + 2 * (120 // 2), 5 + 3 * 4 - 6, (36 - 36 // 3) // 4, 2 * 15 + 3 * (15 + 7)]
print("gold:", gold)

INSTR = "Реши задачу. В конце напиши ответ в формате «Ответ: число»."
Q_LABEL, S_LABEL, A_LABEL = "Задача", "Решение", "Ответ"
COT_PHRASE = "Давай рассуждать шаг за шагом."
EXAMPLES = [   # (задача, шаги решения, ответ) для few-shot
    ("В корзине было 12 яблок. Съели 5, потом положили ещё 4. Сколько яблок в корзине?", "12 - 5 = 7, 7 + 4 = 11.", 11),
    ("Тетрадь стоит 3 тысячи сумов, ручка — 2 тысячи. Сколько тысяч стоят 4 тетради и 2 ручки?", "4 · 3 = 12, 2 · 2 = 4, 12 + 4 = 16.", 16),
]


MODES = ["zero-shot", "few-shot", "zero-shot CoT", "few-shot CoT"]

def make_prompt(question, mode):
    parts = [INSTR]
    if mode.startswith("few-shot"):
        for q, steps, a in EXAMPLES:
            if "CoT" in mode:
                parts.append(f"{Q_LABEL}: {q}\n{S_LABEL}: {steps} {A_LABEL}: {a}")
            else:
                parts.append(f"{Q_LABEL}: {q}\n{A_LABEL}: {a}")       # few-shot без рассуждения: только ответ
    if mode == "zero-shot CoT":
        tail = f"{S_LABEL}: {COT_PHRASE}"
    elif "CoT" in mode:
        tail = f"{S_LABEL}:"
    else:
        tail = f"{A_LABEL}:"
    parts.append(f"{Q_LABEL}: {question}\n{tail}")
    return "\n\n".join(parts)

for mode in MODES:
    print(f"==================== {mode} ====================")
    print(make_prompt(problems[1], mode))
    print()

print(f"{'':<15}{'символов':>9}{'слов':>8}")
for mode in MODES:
    p = make_prompt(problems[1], mode)
    print(f"{mode:<15}{len(p):9d}{len(p.split()):8d}")

Ниже — **заготовленные ответы**, написанные вручную для иллюстрации (это не вывод модели). Они типичны для небольшой модели: без рассуждения модель часто «угадывает» число, с рассуждением чаще отвечает верно, но может ошибиться в арифметике.

У рассуждения есть ещё один плюс: его можно **проверить**. Функция `check_steps` находит в тексте равенства вида «3 · 22 = 66» и пересчитывает их, а `extract_answer` достаёт итоговое число после «Ответ:».

In [ ]:
# ЗАГОТОВЛЕННЫЕ ответы (написаны вручную для иллюстрации, это не вывод модели)
prepared = {
    (1, "zero-shot"): "Ответ: 600",
    (1, "few-shot"): "Ответ: 360",
    (1, "zero-shot CoT"): "Взрослые билеты: 3 · 120 = 360. Детский билет в два раза дешевле: 120 / 2 = 60. "
                          "Детские билеты: 2 · 60 = 120. Всего: 360 + 120 = 480. Ответ: 480",
    (1, "few-shot CoT"): "3 · 120 = 360, 120 / 2 = 60, 2 · 60 = 120, 360 + 120 = 480. Ответ: 480",
    (4, "zero-shot"): "Ответ: 66",
    (4, "few-shot"): "Ответ: 75",
    (4, "zero-shot CoT"): "Виноград стоит 15 + 7 = 22 тысячи. Яблоки: 2 · 15 = 30. Виноград: 3 · 22 = 56. "
                          "Всего: 30 + 56 = 86. Ответ: 86",
    (4, "few-shot CoT"): "15 + 7 = 22, 2 · 15 = 30, 3 · 22 = 66, 30 + 66 = 96. Ответ: 96",
}
ANSWER_RE = r"Ответ[:\s*]*(-?\d+)"


def extract_answer(text):
    # число после «Ответ:»; если его нет — последнее число в тексте
    m = re.findall(ANSWER_RE, text, flags=re.IGNORECASE)
    if m:
        return int(m[-1])
    nums = re.findall(r"-?\d+", text)
    return int(nums[-1]) if nums else None

OPS = {"+": lambda a, b: a + b, "-": lambda a, b: a - b, "·": lambda a, b: a * b,
       "*": lambda a, b: a * b, "×": lambda a, b: a * b, "/": lambda a, b: a / b}

def check_steps(text):
    # находим равенства «a op b = c» и пересчитываем их
    bad = []
    for a, op, b, c in re.findall(r"(\d+)\s*([-+·*×/])\s*(\d+)\s*=\s*(\d+)", text):
        if OPS[op](int(a), int(b)) != int(c):
            bad.append(f"{a} {op} {b} = {c} (должно быть {OPS[op](int(a), int(b)):g})")
    return bad

for (i, mode), text in prepared.items():
    pred = extract_answer(text)
    steps = check_steps(text)
    verdict = "OK " if pred == gold[i] else "ОШИБКА"
    print(f"задача {i}  {mode:<14} {verdict}  {A_LABEL.lower()}={pred:<4} gold={gold[i]:<4} "
          + (f"неверный шаг: {steps}" if steps else ""))

### Как это выглядит на настоящей модели

`Qwen/Qwen2.5-0.5B-Instruct` решает пять задач с четырьмя видами подсказок (жадное декодирование, до 256 новых токенов). На CPU это займёт несколько минут, с GPU («Среда выполнения → Сменить среду выполнения → T4») — около минуты. Модель маленькая: ошибок будет много, а результат на 5 задачах очень шумный, поэтому смотрите на сами ответы, а не только на счёт. Малые модели заметно слабее на узбекском языке, чем на русском и английском: сравните с узбекской версией этого практикума.

In [ ]:
# все задачи × все виды подсказок; модель загружается, если её ещё нет
try:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    if "llm" not in globals():
        name = "Qwen/Qwen2.5-0.5B-Instruct"
        tok = AutoTokenizer.from_pretrained(name)
        device = "cuda" if torch.cuda.is_available() else "cpu"
        dtype = "auto" if device == "cuda" else torch.float32     # на CPU float32 быстрее, чем bfloat16
        llm = AutoModelForCausalLM.from_pretrained(name, torch_dtype=dtype).to(device)

    def ask(prompt, max_new_tokens=256, **gen):
        text = tok.apply_chat_template([{"role": "user", "content": prompt}],
                                       tokenize=False, add_generation_prompt=True)
        inputs = tok([text], return_tensors="pt").to(llm.device)
        out = llm.generate(**inputs, max_new_tokens=max_new_tokens, pad_token_id=tok.eos_token_id, **gen)
        return [tok.decode(o[inputs.input_ids.shape[1]:], skip_special_tokens=True) for o in out]

    score = {m: 0 for m in MODES}
    for i, (q, g) in enumerate(zip(problems, gold)):
        for mode in MODES:
            answer = ask(make_prompt(q, mode), do_sample=False)[0]      # жадное декодирование
            pred = extract_answer(answer)
            score[mode] += int(pred == g)
            print(f"{i} {mode:<14} pred={pred} gold={g}  " + answer.replace("\n", " ")[:110])
    print()
    for mode in MODES:
        print(f"{mode:<14} {score[mode]} / {len(problems)}")
except Exception as e:
    print("Не удалось запустить модель (нужен интернет в Colab):", e)

## 4. Self-consistency: голосование нескольких рассуждений

Вместо одного жадного ответа сгенерируем $N$ рассуждений с сэмплированием ($T>0$) и выберем самый частый итоговый ответ. Пусть каждое рассуждение независимо верно с вероятностью $p$. Если все неверные ответы одинаковы (худший случай: всего два варианта ответа), то для нечётного $N$

$$P_{\text{maj}}(N,p)=\sum_{k=(N+1)/2}^{N}\binom{N}{k}\,p^k(1-p)^{N-k}.$$

При $p>0.5$ точность растёт с $N$ к 1, при $p<0.5$ — падает к 0. На практике неверные ответы обычно **разные**, их голоса «распыляются», и голосование по относительному большинству помогает даже при $p<0.5$. Проверим формулу симуляцией (ничьи разбиваем случайно).

Оговорка: рассуждения одной модели не независимы (модель повторяет одни и те же ошибки), поэтому реальный выигрыш меньше, чем по формуле.


В конце ячейки — то же на «текстовых» путях рассуждения: каждый путь заканчивается строкой «Ответ: …», ответ извлекаем функцией `extract_answer` и голосуем. Пути смоделированы: верный ответ с вероятностью 0.6, иначе одна из типичных ошибок.

In [ ]:
from math import comb
from scipy.stats import binom

def p_majority(N, p):
    # точность голосования большинством, N нечётное, два варианта ответа
    return sum(comb(N, k) * p ** k * (1 - p) ** (N - k) for k in range((N + 1) // 2, N + 1))

assert np.isclose(p_majority(3, 0.6), 3 * 0.6 ** 2 * 0.4 + 0.6 ** 3)          # = 0.648
assert all(np.isclose(p_majority(N, 0.6), binom.sf((N - 1) // 2, N, 0.6)) for N in [1, 5, 11])

def simulate(N, p, m_wrong, trials=20000, rng=None):
    # каждый путь верен с вероятностью p, иначе одна из m_wrong ошибок (равновероятно)
    probs = [p] + [(1 - p) / m_wrong] * m_wrong
    answers = rng.choice(m_wrong + 1, size=(trials, N), p=probs)             # 0 = верный ответ
    counts = (answers[:, :, None] == np.arange(m_wrong + 1)).sum(axis=1)
    winner = np.argmax(counts + 0.5 * rng.random(counts.shape), axis=1)     # случайный разрыв ничьих
    return float(np.mean(winner == 0))

rng = np.random.default_rng(0)
Ns = [1, 3, 5, 7, 9, 11, 15, 21]
table = []
print(f"{'N':>3}{'p=0.6 формула':>18}{'симул.':>7}{'p=0.4 формула':>18}{'симул.':>7}{'p=0.4, 5 ошибок':>22}")
for N in Ns:
    row = (N, p_majority(N, 0.6), simulate(N, 0.6, 1, rng=rng), p_majority(N, 0.4),
           simulate(N, 0.4, 1, rng=rng), simulate(N, 0.4, 5, rng=rng))
    table.append(row)
    print(f"{row[0]:>3}{row[1]:>18.3f}{row[2]:>7.3f}{row[3]:>18.3f}{row[4]:>7.3f}{row[5]:>22.3f}")

table = np.array(table)
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.plot(table[:, 0], table[:, 1], "-", color="#2B55C9", label="p=0.6, формула")
ax.plot(table[:, 0], table[:, 2], "o", color="#2B55C9", label="p=0.6, симул.")
ax.plot(table[:, 0], table[:, 3], "-", color="#C0392B", label="p=0.4, формула")
ax.plot(table[:, 0], table[:, 4], "o", color="#C0392B", label="p=0.4, симул.")
ax.plot(table[:, 0], table[:, 5], "s--", color="#17907E", label="p=0.4, 5 ошибок (симул.)")
ax.axhline(0.5, color="grey", lw=0.5)
ax.set_xlabel("число рассуждений N")
ax.set_ylabel("точность голосования")
ax.set_title("Self-consistency")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()


# смоделированные рассуждения для задачи 1 (верный ответ 480)
wrong_answers = [600, 360, 540, 420]

def sample_path(rng, p=0.6):
    a = gold[1] if rng.random() < p else int(rng.choice(wrong_answers))
    return f"... {A_LABEL}: {a}"

rng = np.random.default_rng(5)
for N in [1, 5, 15]:
    paths = [sample_path(rng) for _ in range(N)]
    votes = Counter(extract_answer(t) for t in paths)
    winner = votes.most_common(1)[0][0]
    print(f"N={N:<3} голоса: {dict(votes)}  ->  {winner}  ({'OK' if winner == gold[1] else 'ОШИБКА'})")

### Как это выглядит на настоящей модели

Семь рассуждений `Qwen/Qwen2.5-0.5B-Instruct` для задачи про рынок (zero-shot CoT, $T=0.8$, top-p = 0.95) и голосование. Ячейка работает только в Colab с интернетом.

In [ ]:
# self-consistency на настоящей модели: 7 сэмплированных рассуждений
try:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    if "llm" not in globals():
        name = "Qwen/Qwen2.5-0.5B-Instruct"
        tok = AutoTokenizer.from_pretrained(name)
        device = "cuda" if torch.cuda.is_available() else "cpu"
        dtype = "auto" if device == "cuda" else torch.float32     # на CPU float32 быстрее, чем bfloat16
        llm = AutoModelForCausalLM.from_pretrained(name, torch_dtype=dtype).to(device)

    k = 4
    text = tok.apply_chat_template([{"role": "user", "content": make_prompt(problems[k], "zero-shot CoT")}],
                                   tokenize=False, add_generation_prompt=True)
    inputs = tok([text], return_tensors="pt").to(llm.device)
    torch.manual_seed(0)
    out = llm.generate(**inputs, max_new_tokens=256, do_sample=True, temperature=0.8, top_p=0.95,
                       num_return_sequences=7, pad_token_id=tok.eos_token_id)
    paths = [tok.decode(o[inputs.input_ids.shape[1]:], skip_special_tokens=True) for o in out]
    preds = [extract_answer(t) for t in paths]
    for t, a in zip(paths, preds):
        print(f"[{a}] " + t.replace("\n", " ")[:120])
    votes = Counter(preds)
    print("\nголоса:", dict(votes), " ->", votes.most_common(1)[0][0], "  gold =", gold[k])
except Exception as e:
    print("Не удалось запустить модель (нужен интернет в Colab):", e)

## 5. Задания

1. **min-p.** Реализуйте фильтр min-p: оставить токены с $p_i \ge \text{min\_p}\cdot \max_j p_j$ и перенормировать. Сравните число оставшихся токенов у min-p = 0.1 и top-p = 0.9 на логитах из раздела 1 при $T = 1$ и $T = 2$. Какой фильтр сильнее реагирует на температуру?
2. **Штраф за повтор.** Добавьте в `generate` параметр `penalty`: из логитов слов, которые уже есть в `out`, вычитается `penalty`. Посчитайте долю различных предложений и среднюю длину предложения при `penalty` = 0, 1, 3 для $T = 0.5$.
3. **Своя подсказка.** Напишите шаблоны zero-shot и few-shot (3 примера) для классификации отзывов на «позитивный / негативный» и функцию `parse_label`, которая достаёт метку из свободного ответа («Скорее позитивный», «Метка: негативный.»). Проверьте её на 4 придуманных вами ответах.
4. **Сколько голосов нужно?** По формуле найдите наименьшее нечётное $N$, при котором точность голосования ≥ 0.9, для $p = 0.6$ и $p = 0.55$. Объясните, почему при $p = 0.5$ такого $N$ не существует.
5. **Проверка ответа.** Дополните `check_steps`: пусть она также сообщает об ошибке, если число после «Ответ:» не совпадает с результатом последнего шага. Проверьте на всех заготовленных ответах и на строке «2 · 15 = 30, 30 + 66 = 96. Ответ: 69».

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь